### Claude Science

$ claude-science

### Tasks and Contexts

Series GSE149884 is a melanoma study using Mus musculus species
URL: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE149884

Study: Series GSE149884 is a melanoma study using the Mus musculus species
Platform: GPL18480 Illumina HiSeq 1500

Context: This melanoma animal model has 4 subtypes: melan (melanocytes), 4c, 4c11minus (tumor not metastatic), and 4c11plus(tumor metastatic)
Replicates: 2 melan, 3 4c, 3 4c11minus, and 3 4c11plus


Tasks:
1) Align the reads according to the last Mus mus reference
2) Do the quality control - remove bad samples, if necessary - document it
3) Are there batch effects? If so, correct the batch
4) Filter the most variable genes
5) Make a CPM table: count reads
6) calc the 3 LFC tables: 4c, 4c11minus and 4c11plus all relative to melan
7) Use BayesPrism to convert bulk transcriptomics to single-cell - find a single-cell study as reference
8) Call all LFC, again, per melanoma type per cell type


For each step:
1) document it
2) develop a script; if possible, build a Nextflow algorithm; otherwise, a single Python/R Script
3) run it - document the input and output

Do it all using Python and uv, besides Nextflow, which uses Java.
When possible, call RScript - create a conda environment for it

In [13]:
import json
res = json.load(open("handoff/gse149884.json"))

print(res.keys())

dict_keys(['n_requested', 'records'])


### Samples

In [14]:
import urllib.request, urllib.parse, json, time

In [15]:
res = json.load(open("handoff/gse149884.json"))
rec = res["records"][0]
samples = rec["samples"]
print("n_samples:", len(samples))
print("series suppl files:", rec.get("supplementary_files"))
print()
for s in samples:
    print(s["accession"], "|", s["title"], "|", s["source_name"])
    for f in s.get("supplementary_files", []):
        print("    ", f)



n_samples: 11
series suppl files: {'series': ['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE149nnn/GSE149884/suppl/GSE149884_RAW.tar'], 'samples': {'GSM4516333': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516333/suppl/GSM4516333_melan2.norm.counts.txt.gz'], 'GSM4516334': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516334/suppl/GSM4516334_melan3.norm.counts.txt.gz'], 'GSM4516335': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516335/suppl/GSM4516335_4c1.norm.counts.txt.gz'], 'GSM4516336': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516336/suppl/GSM4516336_4c2.norm.counts.txt.gz'], 'GSM4516337': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516337/suppl/GSM4516337_4c3.norm.counts.txt.gz'], 'GSM4516338': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516338/suppl/GSM4516338_4c11minus1.norm.counts.txt.gz'], 'GSM4516339': ['ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4516nnn/GSM4516339/suppl/GSM4516339_4c11minus2.norm.counts.txt.gz

### Bioproject

In [16]:
res = json.load(open("handoff/gse149884.json"))
rec = res["records"][0]
print("bioproject:", rec.get("bioproject"))
print("keys:", [k for k in rec.keys()])
for k in ("relations","sra","bioproject","pubmed_ids"):
    print(k, "->", rec.get(k))

bioproject: None
keys: ['accession', 'title', 'organism', 'series_type', 'status', 'submission_date', 'last_update_date', 'summary', 'overall_design', 'pubmed_ids', 'platforms', 'n_samples', 'samples', 'supplementary_files', 'esummary']
relations -> None
sra -> None
bioproject -> None
pubmed_ids -> ['32831131', '33845354', '35075772']


### GSE149884

In [17]:
def get(url):
    req = urllib.request.Request(url, headers={"User-Agent":"python-urllib"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read().decode()

# 1) find SRA study accession via ENA cross-ref search on the GEO series (BioProject)
# First: eutils esearch SRA for the GSE
base="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
r = get(base+"esearch.fcgi?db=sra&term=GSE149884&retmax=50")
print("esearch sra:", r[:500])

esearch sra: <?xml version="1.0" encoding="UTF-8" ?>
<!DOCTYPE eSearchResult PUBLIC "-//NLM//DTD esearch 20060628//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20060628/esearch.dtd">
<eSearchResult><Count>0</Count><RetMax>0</RetMax><RetStart>0</RetStart><IdList/><TranslationSet/><QueryTranslation>(GSE149884[All Fields])</QueryTranslation><ErrorList><PhraseNotFound>GSE149884</PhraseNotFound></ErrorList><WarningList><OutputMessage>No items found.</OutputMessage></WarningList></eSearchResult>



### ENA: resolve GEO sample GSM to SRA via ENA cross-reference search

In [18]:
def get(url):
    req = urllib.request.Request(url, headers={"User-Agent":"python-urllib"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read().decode()

# ENA: resolve GEO sample GSM to SRA via ENA cross-reference search
# Use ENA portal: find run(s) linked to the study. Try searching by sample_accession=GSM won't work.
# Use NCBI elink from GEO DataSets (gds) uid. First esearch gds for the GSE to get uid.
base="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
r = get(base+"esearch.fcgi?db=gds&term=GSE149884[ACCN]&retmax=5")
print(r[:1200])

<?xml version="1.0" encoding="UTF-8" ?>
<!DOCTYPE eSearchResult PUBLIC "-//NLM//DTD esearch 20060628//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20060628/esearch.dtd">
<eSearchResult><Count>13</Count><RetMax>5</RetMax><RetStart>0</RetStart><IdList>
<Id>200149884</Id>
<Id>100018480</Id>
<Id>304516343</Id>
<Id>304516342</Id>
<Id>304516341</Id>
</IdList><TranslationSet/><TranslationStack>   <TermSet>    <Term>GSE149884[ACCN]</Term>    <Field>ACCN</Field>    <Count>13</Count>    <Explode>N</Explode>   </TermSet>   <OP>GROUP</OP>  </TranslationStack><QueryTranslation>GSE149884[ACCN]</QueryTranslation></eSearchResult>



In [19]:
def get(url):
    req = urllib.request.Request(url, headers={"User-Agent":"python-urllib"})
    with urllib.request.urlopen(req, timeout=90) as r:
        return r.read().decode()

# ENA portal: search runs by GEO series via the study cross-reference.
# Try: find the SRA study using ENA text search on the GEO accession.
url = ("https://www.ebi.ac.uk/ena/portal/api/search?"
       "result=read_run&query=study_accession%3D%22GSE149884%22&fields=run_accession&format=tsv")
# GEO accession won't be a study_accession; instead use the GEO->SRA linking through NCBI elink bioproject
base="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
# elink gds uid -> bioproject
r = get(base+"elink.fcgi?dbfrom=gds&db=bioproject&id=200149884")
print(r[:1500])

HTTPError: HTTP Error 500: Internal Server Error

In [20]:
def get(url):
    req = urllib.request.Request(url, headers={"User-Agent":"python-urllib"})
    with urllib.request.urlopen(req, timeout=90) as r:
        return r.read().decode()
base="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
# bioproject uid 630432 -> get accession
r = get(base+"esummary.fcgi?db=bioproject&id=630432")
import re
acc = re.search(r'<Project_Acc>(PRJ\w+)</Project_Acc>', r)
print("bioproject acc:", acc.group(1) if acc else "?")
print(r[:800])

bioproject acc: PRJNA630432
<?xml version="1.0" encoding="UTF-8" ?>
<!DOCTYPE eSummaryResult PUBLIC "-//NLM//DTD esummary bioproject 20140903//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20140903/esummary_bioproject.dtd">
<eSummaryResult>
<DocumentSummarySet status="OK">
<DbBuild>Build260706-0715.1</DbBuild>

<DocumentSummary uid="630432">
	<TaxId>10090</TaxId>
	<Project_Id>630432</Project_Id>
	<Project_Acc>PRJNA630432</Project_Acc>
	<Project_Type>Primary submission</Project_Type>
	<Project_Data_Type>Transcriptome or Gene expression</Project_Data_Type>
	<Sort_By_ProjectType>618150</Sort_By_ProjectType>
	<Sort_By_DataType>604620</Sort_By_DataType>
	<Sort_By_Organism>455444</Sort_By_Organism>
	<Project_Subtype></Project_Subtype>
	<Project_Target_Scope>Multiisolate</Project_Target_Scope>
	<Project_Target_Material>Tr
